In [1]:
import pandas as pd

df = pd.read_csv("dataset/creditcard_2023.csv")  # check file
# V1-V28: Anonymized features representing various transaction attributes (e.g., time, location, etc.)
df.head()

,id,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0,-0.260648,-0.469648,2.496266,-0.083724,0.129681,0.732898,0.519014,-0.130006,0.727159,...,-0.110552,0.217606,-0.134794,0.165959,0.126280,-0.434824,-0.081230,-0.151045,17982.10,0
1,1,0.985100,-0.356045,0.558056,-0.429654,0.277140,0.428605,0.406466,-0.133118,0.347452,...,-0.194936,-0.605761,0.079469,-0.577395,0.190090,0.296503,-0.248052,-0.064512,6531.37,0
2,2,-0.260272,-0.949385,1.728538,-0.457986,0.074062,1.419481,0.743511,-0.095576,-0.261297,...,-0.005020,0.702906,0.945045,-1.154666,-0.605564,-0.312895,-0.300258,-0.244718,2513.54,0
3,3,-0.152152,-0.508959,1.746840,-1.090178,0.249486,1.143312,0.518269,-0.065130,-0.205698,...,-0.146927,-0.038212,-0.214048,-1.893131,1.003963,-0.515950,-0.165316,0.048424,5384.44,0
4,4,-0.206820,-0.165280,1.527053,-0.448293,0.106125,0.530549,0.658849,-0.212660,1.049921,...,-0.106984,0.729727,-0.161666,0.312561,-0.414116,1.071126,0.023712,0.419117,14278.97,0


In [2]:
# Define features (X) and target (y)
# Drop 'id' (identifier) and 'Class' (target) for features
feature_cols = [c for c in df.columns if c not in ['id', 'Class']]
X = df[feature_cols]
y = df['Class']

# Split into train (80%) and test (20%)
# stratify=y keeps the same fraud/non-fraud ratio in both sets (important for imbalanced data)
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(f"Train: {X_train.shape[0]} samples, Test: {X_test.shape[0]} samples")
print(f"Train fraud rate: {y_train.mean():.4f}, Test fraud rate: {y_test.mean():.4f}")

Train: 454904 samples, Test: 113726 samples
Train fraud rate: 0.5000, Test fraud rate: 0.5000


### Regularized QDA

In [3]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("qda", QuadraticDiscriminantAnalysis())
])

Grid Search란

모델 파라미터 vs 하이퍼파라미터 </br>
① 모델 파라미터
- Logistic의 w, b
- LDA/QDA의 평균, 공분산
- 이건 fit()이 찾는 값이야.

② 하이퍼파라미터
- Logistic의 C
- LDA의 shrinkage
- QDA의 reg_param

이건 우리가 직접 정해야 하는 값. fit은 이 값을 조정하지 않음

그럼 grid search는 뭐 하냐?

Grid Search는 이렇게 함:
```
reg_param = 0.0 → CV 점수 계산
reg_param = 0.1 → CV 점수 계산
reg_param = 0.2 → CV 점수 계산
...
```

그 중 가장 CV 점수가 높은 reg_param을 선택하는 것. 즉 모델의 하이퍼파라미터를 선택하는 과정

그럼 fit이랑 뭐가 다르냐?
| 역할          | 하는 일                      |
| ----------- | ------------------------- |
| fit         | 주어진 reg_param으로 평균/공분산 계산 |
| grid search | 어떤 reg_param이 제일 좋은지 찾기   |

- reg_param = 공부 방법 선택
- grid search = 공부 방법 여러 개 실험해보고 제일 좋은 방법 고르기

그래서 전체 흐름은 이렇게 된다
```
1. reg_param 후보 설정
2. 각 후보에 대해 CV
3. 가장 좋은 reg_param 선택
4. 선택된 reg_param으로 train 전체 fit
5. test 평가
```

In [4]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
import numpy as np

param_grid = {
    "qda__reg_param": np.linspace(0, 0.5, 6)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    pipe,
    param_grid,
    cv=cv,
    scoring="average_precision"
)

grid.fit(X_train, y_train)

print("Best reg_param:", grid.best_params_)
print("Best CV score:", grid.best_score_)

Best reg_param: {'qda__reg_param': np.float64(0.1)}
Best CV score: 0.9898402177074794


In [5]:
best_model = grid.best_estimator_
best_model.fit(X_train, y_train)

,steps,"[('scaler', ...), ('qda', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,priors,None
,reg_param,np.float64(0.1)
,store_covariance,False
,tol,0.0001


In [6]:
from sklearn.metrics import roc_auc_score, average_precision_score

y_prob = best_model.predict_proba(X_test)[:, 1]

print("ROC:", roc_auc_score(y_test, y_prob))
print("PR:", average_precision_score(y_test, y_prob))

ROC: 0.9889256875165914
PR: 0.9887920986133145
